# 33 — Analisis gabungan hasil eksperimen


In [1]:
from pathlib import Path
import os, types
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', force_remount=False)
_explicit = os.environ.get('ANALISIS33_ROOT') or os.environ.get('CARE_KM_ROOT')
_options = ([Path(_explicit)] if _explicit else []) + [
    Path('/content/drive/MyDrive/REVISED'), Path.cwd(), Path.cwd().parent]
ROOT_33 = next((p.resolve() for p in _options if all((p/d).is_dir()
    for d in ('LIB', 'PYTHON', 'OUTPUT'))), None)
if ROOT_33 is None:
    raise FileNotFoundError('Letakkan LIB, PYTHON dan OUTPUT di MyDrive/REVISED.')
print('Proyek:', ROOT_33)


Mounted at /content/drive
Proyek: /content/drive/MyDrive/REVISED


## Isi dan integritas sumber

Tahap 25–32 harus berstatus selesai dengan protokol penuh. Manifest keluaran
26–32 dan manifest artefak 32 diperiksa berdasarkan SHA-256, termasuk berkas
model sebagai byte tanpa membukanya sebagai objek model. Prediksi alias dari
26/27/28/29/30 harus sama dan hanya satu salinan dihitung dalam analisis.

Sumber yang hilang/berubah menghentikan proses dengan lokasi berkas. Notebook
33 tidak memperbaiki hasil lama dengan melakukan fitting otomatis. Pemeriksaan
ribuan berkas di Drive dapat memerlukan beberapa menit; kemajuan ditampilkan.
Cache 33 juga memeriksa isi sumber, kode, versi pustaka dan seluruh hasil.


In [2]:
KONFIGURASI_33 = {'force': False, 'figures': True}


## Protokol dan inferensi

Lima konteks dilaporkan terpisah: holdout Pendikom 139, holdout LMS 1.998,
split kursus Pendikom 461, temporal Pendikom 115, serta split kursus LMS 6.658.
Jumlah tersebut diverifikasi dari sumber, bukan digunakan untuk membuat data.
Peserta lintas protokol dapat tumpang tindih; jangan menggabungkannya sebagai
dataset independen. LMS mengukur kesesuaian terhadap teacher CARE-KM yang
dipelajari dari train, sedangkan Pendikom dibandingkan dengan ILS independen.

Metrik dan interval memakai prediksi/bobot bootstrap yang sudah tersimpan.
Empat dimensi dipertahankan bersama. Untuk initializer stokastik, hitung metrik
per run lalu rerata; tidak memilih seed terbaik atau mengambil voting. Prediksi
unresolved tetap merupakan abstention/kesalahan. Referensi unresolved hanya
dikecualikan dari metrik yang membutuhkan kutub referensi; coverage dilaporkan.

Bank 10.000 draw dipakai kembali, bukan membuat bootstrap baru. Split lama
memakai unit peserta; split kursus/waktu memakai unit kursus. Interval bersyarat
pada fitted folds/runs yang ada. Interval keluarga mean BA menggunakan
Bonferroni per keluarga; metrik lain deskriptif. Ini tidak membuktikan ekuivalensi.

Benchmark memakai sepuluh dataset independen, dengan rerata 30 matched
resamples per dataset. Kernel statistik asli dibekukan: Friedman/Kendall W,
exact signed-rank enumeration, average tied ranks, zero tolerance 1e-12,
rank-biserial, serta Holm untuk tujuh kontras CARE-KM pada DBI dan SI terpisah.
Tidak menambahkan pengujian semua pasangan atau menganggap resample independen.


In [3]:
MATH33_SOURCE = 'def count_metrics(counts):\n    """2 x 3 confusion counts; column 3 is abstention, never dropped."""\n    a, b, c, d, e, f = np.moveaxis(np.asarray(counts, float), -1, 0)\n    pos = a + b + c\n    neg = d + e + f\n    total = pos + neg\n    with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n        accuracy = (a + e) / total\n        ba = 0.5 * (a / pos + e / neg)\n        denp = 2 * a + d + b + c\n        denn = 2 * e + b + d + f\n        fp = np.divide(2 * a, denp, out=np.zeros_like(a), where=denp > 0)\n        fn = np.divide(2 * e, denn, out=np.zeros_like(e), where=denn > 0)\n    return np.stack([accuracy, ba, 0.5 * (fp + fn)], axis=-1)\n\ndef confusion_masks(truth, prediction):\n    t = truth[None, :, :]\n    p = prediction\n    conditions = [(t == 1) & (p == 1), (t == 1) & (p == -1), (t == 1) & (p == 0), (t == -1) & (p == 1), (t == -1) & (p == -1), (t == -1) & (p == 0)]\n    return np.stack(conditions, axis=-1).transpose(1, 0, 2, 3).astype(float)\n\nMETHODS = [\'Random\', \'Huang\', \'Cao\', \'Bai\', \'K-Modes++\', \'HR-2023\', \'CARE-KM\', \'BK-Modes\']\n\nSTOCHASTIC = {\'Random\', \'Huang\', \'K-Modes++\', \'HR-2023\'}\n\nMETRICS = [\'initial_cost\', \'final_cost\', \'dbi\', \'si\', \'iterations\']\n\ndef exact_signed_rank(differences, zero_tolerance=1e-12):\n    differences = np.asarray(differences, dtype=float)\n    if not np.isfinite(differences).all():\n        raise ValueError(\'Paired differences mengandung nilai tidak valid.\')\n    nonzero = np.abs(differences) > zero_tolerance\n    values = differences[nonzero]\n    if not len(values):\n        return {\'statistic\': 0.0, \'p_value\': 1.0, \'rank_biserial\': 0.0, \'n_nonzero\': 0, \'n_zero\': len(differences), \'absolute_ranks\': [], \'positive_rank_sum\': 0.0, \'negative_rank_sum\': 0.0}\n    ranks = pd.Series(np.abs(values)).rank(method=\'average\').to_numpy(float)\n    positive, negative = (float(ranks[values > 0].sum()), float(ranks[values < 0].sum()))\n    observed = min(positive, negative)\n    count = 0\n    for signs in itertools.product((0, 1), repeat=len(values)):\n        pos = float(np.dot(signs, ranks))\n        count += min(pos, ranks.sum() - pos) <= observed + 1e-12\n    return {\'statistic\': observed, \'p_value\': count / 2 ** len(values), \'rank_biserial\': (positive - negative) / (positive + negative), \'n_nonzero\': len(values), \'n_zero\': int((~nonzero).sum()), \'absolute_ranks\': ranks.tolist(), \'positive_rank_sum\': positive, \'negative_rank_sum\': negative}\n\ndef holm(values):\n    order = np.argsort(values, kind=\'stable\')\n    answer, running = (np.zeros(len(values)), 0.0)\n    for rank, index in enumerate(order):\n        running = max(running, (len(values) - rank) * values[index])\n        answer[index] = min(1.0, running)\n    return answer\n\ndef benchmark_statistics(summary, cfg, gd, output):\n    from scipy.stats import friedmanchisquare\n    ranks, friedman, pairwise, audit = ([], [], [], [])\n    notes = []\n    for scope in [\'BENCHMARK_FULL\', \'BENCHMARK_RESAMPLING\']:\n        frame = summary.loc[summary[\'scope\'].eq(scope)]\n        if frame.empty:\n            notes.append(scope + \': belum dijalankan\')\n            continue\n        if scope == \'BENCHMARK_RESAMPLING\' and cfg[\'resample_runs\'] != 30:\n            notes.append(scope + \': pengaturan diagnostik kurang dari 30 resample; statistik utama tidak dihitung\')\n            continue\n        if scope == \'BENCHMARK_FULL\' and cfg[\'benchmark_stochastic_runs\'] != 10:\n            notes.append(scope + \': pengaturan diagnostik bukan 10 run stokastik; statistik utama tidak dihitung\')\n            continue\n        complete = set(frame.dataset) == set(gd.DATASET_ORDER) and len(frame) == len(gd.DATASET_ORDER) * len(METHODS)\n        expected_runs = frame.method.map(lambda method: 30 if scope == \'BENCHMARK_RESAMPLING\' else 10 if method in STOCHASTIC else 1)\n        if not complete or (frame.n_failures > 0).any() or (not frame.n_runs.eq(expected_runs).all()):\n            notes.append(scope + \': statistik sepuluh dataset belum lengkap atau ada kegagalan; tidak dihitung paksa\')\n            continue\n        for metric in METRICS:\n            matrix = frame.pivot(index=\'dataset\', columns=\'method\', values=metric + \'_mean\').reindex(index=gd.DATASET_ORDER, columns=METHODS)\n            if not np.isfinite(matrix.to_numpy(float)).all():\n                notes.append(scope + \'/\' + metric + \': metrik tidak lengkap\')\n                continue\n            ranking = matrix.rank(axis=1, ascending=metric != \'si\', method=\'average\')\n            for dataset in matrix.index:\n                for method in METHODS:\n                    ranks.append({\'source\': scope, \'metric\': metric, \'dataset\': dataset, \'method\': method, \'value\': matrix.loc[dataset, method], \'rank\': ranking.loc[dataset, method]})\n            if (matrix.nunique(axis=1) == 1).all():\n                stat, p = (np.nan, np.nan)\n                notes.append(scope + \'/\' + metric + \': semua metode seri; Friedman tidak terdefinisi\')\n            else:\n                stat, p = friedmanchisquare(*(matrix[method].to_numpy(float) for method in METHODS))\n            friedman.append({\'source\': scope, \'metric\': metric, \'friedman_chi2\': stat, \'p_value\': p, \'kendall_w\': stat / (len(matrix) * (len(METHODS) - 1)), \'n_datasets\': len(matrix), \'n_methods\': len(METHODS)})\n            if scope != \'BENCHMARK_RESAMPLING\' or metric not in (\'dbi\', \'si\'):\n                continue\n            family = []\n            for method in [m for m in METHODS if m != \'CARE-KM\']:\n                difference = matrix[\'CARE-KM\'] - matrix[method] if metric == \'si\' else matrix[method] - matrix[\'CARE-KM\']\n                result = exact_signed_rank(difference.to_numpy(float))\n                for dataset, value in difference.items():\n                    audit.append({\'metric\': metric, \'baseline\': method, \'dataset\': dataset, \'care_value\': matrix.loc[dataset, \'CARE-KM\'], \'baseline_value\': matrix.loc[dataset, method], \'improvement_favoring_care\': value, \'zero\': abs(value) <= 1e-12})\n                family.append({\'metric\': metric, \'baseline\': method, \'wins\': int((difference > 1e-12).sum()), \'ties\': int((difference.abs() <= 1e-12).sum()), \'losses\': int((difference < -1e-12).sum()), **{k: v for k, v in result.items() if k != \'absolute_ranks\'}, \'p_method\': \'exact enumeration of nonzero signed-rank signs\', \'zero_method\': \'discard abs(diff)<=1e-12\', \'absolute_rank_ties\': \'average\', \'alternative\': \'two-sided\'})\n            adjusted = holm([row[\'p_value\'] for row in family])\n            for row, value in zip(family, adjusted):\n                row[\'holm_p\'] = value\n                pairwise.append(row)\n    folder = output / \'STATISTIK_BENCHMARK\'\n    save_csv(folder / \'01_dataset_values_and_ranks.csv\', pd.DataFrame(ranks, columns=[\'source\', \'metric\', \'dataset\', \'method\', \'value\', \'rank\']))\n    rank_frame = pd.DataFrame(ranks)\n    average = rank_frame.groupby([\'source\', \'metric\', \'method\'], as_index=False)[\'rank\'].mean().rename(columns={\'rank\': \'average_rank\'}) if len(rank_frame) else pd.DataFrame(columns=[\'source\', \'metric\', \'method\', \'average_rank\'])\n    save_csv(folder / \'02_average_ranks.csv\', average)\n    save_csv(folder / \'03_friedman_kendall.csv\', pd.DataFrame(friedman, columns=[\'source\', \'metric\', \'friedman_chi2\', \'p_value\', \'kendall_w\', \'n_datasets\', \'n_methods\']))\n    save_csv(folder / \'04_paired_wilcoxon_holm.csv\', pd.DataFrame(pairwise) if pairwise else pd.DataFrame(columns=[\'metric\', \'baseline\', \'wins\', \'ties\', \'losses\', \'statistic\', \'p_value\', \'rank_biserial\', \'holm_p\']))\n    audit_frame = pd.DataFrame(audit)\n    if len(audit_frame):\n        audit_frame[\'absolute_rank\'] = audit_frame.groupby([\'metric\', \'baseline\'])[\'improvement_favoring_care\'].transform(lambda x: x.abs().where(x.abs() > 1e-12).rank(method=\'average\'))\n        audit_frame[\'signed_rank\'] = np.sign(audit_frame.improvement_favoring_care) * audit_frame.absolute_rank\n    save_csv(folder / \'05_paired_differences_and_ranks.csv\', audit_frame if len(audit_frame) else pd.DataFrame(columns=[\'metric\', \'baseline\', \'dataset\', \'care_value\', \'baseline_value\', \'improvement_favoring_care\', \'zero\', \'absolute_rank\', \'signed_rank\']))\n    save_json(folder / \'status_statistik.json\', {\'notes\': notes, \'independent_units\': \'ten datasets\', \'primary_source\': \'means of thirty matched resamples within each dataset\', \'multiple_testing\': \'Holm, seven CARE-KM contrasts separately for DBI and SI\', \'runtime_ranks\': \'not calculated from mixed cached/new timing measurements\'})\n    return notes\n'


In [4]:
RUNTIME33_SOURCE = '"""Read-only synthesis of completed experiments. No estimator is loaded or fitted."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport ast, gzip, hashlib, importlib.metadata, io, itertools, json, platform, time, types, uuid\nimport numpy as np\nimport pandas as pd\n\nSTAGE=\'33_Analisis_Gabungan_Hasil\'\nSTEMS={25:\'25_Perbandingan_Initializer_Terintegrasi\',26:\'26_Pengaruh_Initializer_Profil_Semantik\',\n27:\'27_Perbandingan_Pembentukan_Profil\',28:\'28_Referensi_Supervised_ILS\',29:\'29_Sensitivitas_Jumlah_Cluster_K\',\n30:\'30_Direct_Assignment_vs_Classifier\',31:\'31_Benchmark_Runtime_dan_Failure_Audit\',32:\'32_Transportabilitas_Antarkursus_dan_Waktu\'}\nKEYS=[\'active_reflective\',\'sensing_intuitive\',\'visual_verbal\',\'sequential_global\']\nMETRICS=[\'accuracy\',\'balanced_accuracy\',\'macro_f1\']\nMETHODS=[\'Random\',\'Huang\',\'Cao\',\'Bai\',\'K-Modes++\',\'HR-2023\',\'CARE-KM\',\'BK-Modes\']\nABBR=[(\'Act.\',\'Ref.\'),(\'Sens.\',\'Int.\'),(\'Vis.\',\'Ver.\'),(\'Seq.\',\'Gl.\')]\nPOLES=[(\'Active\',\'Reflective\'),(\'Sensing\',\'Intuitive\'),(\'Visual\',\'Verbal\'),(\'Sequential\',\'Global\')]\nBASE=\'CARE-KM|PRIMARY_MODE_FIRST\'\nCELL_NAMES=[\'true_first_pred_first\',\'true_first_pred_second\',\'true_first_abstain\',\'true_second_pred_first\',\'true_second_pred_second\',\'true_second_abstain\']\n\ndef clean(x):\n    if isinstance(x,dict):return {str(k):clean(v) for k,v in x.items()}\n    if isinstance(x,(list,tuple,np.ndarray)):return [clean(v) for v in x]\n    if isinstance(x,(float,np.floating)):return float(x) if np.isfinite(x) else None\n    if isinstance(x,np.integer):return int(x)\n    if isinstance(x,np.bool_):return bool(x)\n    return x\n\ndef digest(x):return hashlib.sha256(json.dumps(clean(x),sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False).encode()).hexdigest()\n\ndef filehash(p):\n    h=hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for part in iter(lambda:f.read(1024*1024),b\'\'):h.update(part)\n    return h.hexdigest()\n\ndef atomic(p,data):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    temp=p.with_name(p.name+\'.\'+uuid.uuid4().hex[:8]+\'.tmp\')\n    temp.write_bytes(data);temp.replace(p)\n\ndef save_json(p,x):atomic(p,json.dumps(clean(x),ensure_ascii=False,indent=2,allow_nan=False).encode())\n\ndef save_csv(p,f):\n    b=f.to_csv(index=False,lineterminator=\'\\n\',float_format=\'%.17g\').encode()\n    atomic(p,gzip.compress(b,mtime=0) if str(p).endswith(\'.gz\') else b)\n\ndef code_hash(nb):\n    cells=[]\n    for c in nb[\'cells\']:\n        if c[\'cell_type\']!=\'code\':continue\n        s=\'\'.join(c[\'source\'])\n        if \'analysis33_configuration\' in c.get(\'metadata\',{}).get(\'tags\',[]):\n            t=ast.parse(s)\n            if len(t.body)!=1 or not isinstance(t.body[0],ast.Assign) or len(t.body[0].targets)!=1 or not isinstance(t.body[0].targets[0],ast.Name) or t.body[0].targets[0].id!=\'KONFIGURASI_33\' or not isinstance(ast.literal_eval(t.body[0].value),dict):\n                raise RuntimeError(\'Sel konfigurasi 33 hanya boleh memuat literal KONFIGURASI_33.\')\n        else:cells.append(s)\n    return digest(cells)\n\nclass Reader:\n    def __init__(self,root):self.root=Path(root).resolve();self.registry={};self.stamps={};self.checked=0\n    def path(self,rel):\n        p=(self.root/rel).resolve()\n        if not p.is_relative_to(self.root):raise ValueError(\'Lokasi berada di luar REVISED: \'+str(rel))\n        return p\n    def relative(self,p):return Path(p).relative_to(self.root).as_posix()\n    def hash(self,rel):\n        p=self.path(rel);st=p.stat();stamp=(st.st_size,st.st_mtime_ns,st.st_ctime_ns)\n        if self.stamps.get(rel)==stamp:return self.registry[rel]\n        if p.suffix.lower()==\'.ipynb\' and p.stem==STAGE:\n            for attempt in range(10):\n                try:\n                    a=p.stat();nb=json.loads(p.read_text(encoding=\'utf-8-sig\'));b=p.stat()\n                    if (a.st_size,a.st_mtime_ns)!=(b.st_size,b.st_mtime_ns):raise ValueError(\'Autosave berlangsung\')\n                    h=\'code:\'+code_hash(nb);stamp=(b.st_size,b.st_mtime_ns,b.st_ctime_ns);break\n                except (ValueError,OSError,KeyError):\n                    if attempt==9:raise\n                    time.sleep(min(.1*2**attempt,.8))\n        else:h=filehash(p)\n        if rel in self.registry and self.registry[rel]!=h:raise RuntimeError(\'Isi sumber berubah selama analisis: \'+rel)\n        self.registry[rel]=h;self.stamps[rel]=stamp;self.checked+=1\n        if self.checked%1000==0:print(\'Memeriksa integritas sumber:\',self.checked,flush=True)\n        return h\n    def read(self,rel):\n        self.hash(rel);b=self.path(rel).read_bytes();self.hash(rel);return b\n    def json(self,rel):return json.loads(self.read(rel).decode(\'utf-8-sig\'))\n    def frame(self,rel,strings=False):\n        b=self.read(rel)\n        if rel.endswith(\'.gz\'):b=gzip.decompress(b)\n        return pd.read_csv(io.BytesIO(b),dtype=str if strings else None,keep_default_na=not strings,float_precision=\'round_trip\')\n    def bank(self,rel):\n        with np.load(io.BytesIO(self.read(rel)),allow_pickle=False) as a:return {k:a[k].copy() for k in a.files}\n    def unchanged(self):\n        for rel in list(self.registry):self.hash(rel)\n\ndef up(n,rel):return \'OUTPUT/TAMBAHAN/\'+STEMS[n]+\'/\'+rel\n\ndef interval(x,alpha=.05):\n    x=np.asarray(x,float);x=x[np.isfinite(x)]\n    return (float(np.quantile(x,alpha/2)),float(np.quantile(x,1-alpha/2)),len(x)) if len(x) else (np.nan,np.nan,0)\n\ndef normalize_route(p):\n    if p in (\'DIRECT\',\'CARE_DIRECT\',\'CARE-KM_DIRECT\'):return BASE\n    if p in (\'BEHAVIORAL_DIRECT\',\'DIRECT_BEHAVIORAL_RULES\'):return \'DIRECT_BEHAVIORAL_RULES\'\n    if p==\'CARE_MEMBER_ONLY\':return \'CARE-KM|MEMBER_ONLY\'\n    if p==\'CARE_MEMBER_FIRST\':return \'CARE-KM|DIRECTION_CONSISTENT_MEMBER_FIRST\'\n    if p.endswith(\'_DIRECT\') and p[:-7] in METHODS:return p[:-7]+\'|PRIMARY_MODE_FIRST\'\n    if p in (\'NB\',\'DT\',\'SVM\'):return \'CDS_\'+p\n    return p\n\ndef directions(f,columns):\n    arr=[]\n    for i,col in enumerate(columns):\n        s=f[col].astype(str)\n        if s.isin([\'-1\',\'0\',\'1\']).all():a=s.to_numpy(np.int8)\n        else:\n            pos=s.str.startswith((POLES[i][0],ABBR[i][0]));neg=s.str.startswith((POLES[i][1],ABBR[i][1]))\n            unresolved=s.str.upper().isin([\'UNRESOLVED\',\'AMBIGUOUS\',\'UNKNOWN\',\'NONE\',\'NAN\',\'\',\'0\'])\n            if not (pos^neg|unresolved).all():raise RuntimeError(\'Kode arah tidak dikenal: \'+col+\' \'+str(s[~(pos^neg|unresolved)].unique()[:5]))\n            a=np.where(pos,1,np.where(neg,-1,0)).astype(np.int8)\n        arr.append(a)\n    return np.column_stack(arr)\n\ndef verify_sources(r,out):\n    rows=[];statuses={}\n    for n,stem in STEMS.items():\n        if n==25:rel=\'OUTPUT/TAMBAHAN/KONTROL/\'+stem+\'/status_eksekusi.json\'\n        else:rel=up(n,\'KONTROL/status_eksekusi.json\')\n        s=r.json(rel);statuses[str(n)]=s\n        if s.get(\'status\')!=\'selesai\' or s.get(\'full_protocol\') is not True:raise RuntimeError(\'Tahap \'+str(n)+\' belum selesai dengan protokol penuh: \'+rel)\n        r.hash(\'PYTHON/\'+stem+\'.ipynb\')\n        if n>=26:\n            rel=up(n,\'KONTROL/manifest_keluaran.json\');manifest=r.json(rel)\n            for path,want in manifest.items():\n                if r.hash(path)!=want:raise RuntimeError(\'Checksum sumber tidak sesuai: \'+path+\'. Jalankan/periksa tahap sumber; 33 tidak melakukan fitting.\')\n            rows.append({\'stage\':n,\'manifest\':rel,\'references\':len(manifest),\'verified\':True})\n    rel=up(32,\'PROTOKOL/06_manifest_artefak_fit_dan_prediksi.json\');manifest=r.json(rel)\n    for path,want in manifest.items():\n        if r.hash(path)!=want:raise RuntimeError(\'Artefak tahap 32 berubah: \'+path)\n    rows.append({\'stage\':32,\'manifest\':rel,\'references\':len(manifest),\'verified\':True})\n    # Only the active libraries are catalogued; raw DATA is not needed by this analysis.\n    for p in sorted((r.root/\'LIB\').rglob(\'*\')):\n        if p.is_file() and \'__pycache__\' not in p.parts and p.suffix.lower() in (\'.py\',\'.csv\',\'.json\',\'.txt\'):r.hash(r.relative(p))\n    save_csv(out/\'AUDIT/01_integritas_tahap_25_32.csv\',pd.DataFrame(rows))\n    save_json(out/\'AUDIT/02_status_sumber.json\',statuses)\n    return statuses\n\ndef comparison_alias(r,a,b,label,records):\n    cols=[\'identity\',\'run\']+[\'reference_\'+k for k in KEYS]+[\'predicted_\'+k for k in KEYS]\n    def norm(f):return f[cols].astype(str).sort_values([\'identity\',\'run\']).reset_index(drop=True)\n    aa,bb=norm(a),norm(b)\n    if not aa.equals(bb):raise RuntimeError(\'Prediksi alias berbeda: \'+label)\n    records.append({\'alias\':label,\'rows\':len(aa),\'same_predictions_and_reference\':True,\'analysis_use\':\'one canonical copy\'})\n\ndef metric_bank(a,index):\n    if list(a[\'metric_names\'])!=METRICS or list(a[\'dimensions\']) not in (KEYS+[\'mean_four_dimensions\'],KEYS+[\'OVERALL\']):raise RuntimeError(\'Urutan dimensi/metrik bank tidak dikenal.\')\n    z=np.asarray(a[\'metrics\'][:,index],float)\n    if z.ndim!=3 or z.shape[1:]!=(5,3):raise RuntimeError(\'Ukuran bank metrik tidak sesuai.\')\n    return z\n\ndef canonical_contexts(r):\n    aliases=[];contexts=[]\n    f32=r.frame(up(32,\'PREDIKSI/01_semua_prediksi.csv.gz\'),True)\n    f32[\'run\']=f32.run.astype(int);f32[\'fold\']=f32.fold.astype(int)\n    for k in KEYS:f32[k]=f32[k].astype(np.int8)\n    ref=r.frame(\'LIB/PROTOKOL/pendikom_ils_labels.csv\',True)\n    if ref.userid.duplicated().any():raise RuntimeError(\'ID ILS ganda.\')\n    refs=dict(zip(ref.userid,directions(ref,[\'ils_\'+k+\'_code\' for k in KEYS]).tolist()))\n    course_ref=dict(zip(ref.userid,ref.courseid))\n    lms_course=f32.loc[f32.cohort.eq(\'LMS\'),[\'identity\',\'course\']].drop_duplicates()\n    if lms_course.identity.duplicated().any():raise RuntimeError(\'Peserta LMS memiliki lebih dari satu kursus terpilih.\')\n    lms_course=dict(zip(lms_course.identity,lms_course.course))\n    old=r.frame(up(27,\'ILS/01_prediksi_per_peserta.csv.gz\'),True);old[\'run\']=old.run.astype(int)\n    old[\'_prediction_source\']=up(27,\'ILS/01_prediksi_per_peserta.csv.gz\');old[\'_bank_source\']=up(27,\'ILS/06_bootstrap_paired_metrics.npz\')\n    p26=r.frame(up(26,\'ILS/01_prediksi_per_peserta.csv\'),True);p26[\'run\']=p26.run.astype(int)\n    seedlookup={(x.method,int(x.run)):str(x.seed) for x in p26[[\'method\',\'run\',\'seed\']].drop_duplicates().itertuples(index=False)}\n    old[\'seed\']=[seedlookup.get((str(path).split(\'|\')[0],int(run)),\'\') for path,run in old[[\'pathway\',\'run\']].itertuples(index=False,name=None)]\n    for method in METHODS:\n        comparison_alias(r,old.loc[old.pathway.eq(method+\'|PRIMARY_MODE_FIRST\')],p26.loc[p26.method.eq(method)],\'26/27 \'+method,aliases)\n    a27=r.bank(up(27,\'ILS/06_bootstrap_paired_metrics.npz\'));a26=r.bank(up(26,\'ILS/06_bootstrap_paired_metrics.npz\'))\n    banks={p:metric_bank(a27,i) for i,p in enumerate(a27[\'pathways\'])}\n    for i,m in enumerate(a26[\'methods\']):\n        if not np.array_equal(metric_bank(a26,i),banks[m+\'|PRIMARY_MODE_FIRST\'],equal_nan=True):raise RuntimeError(\'Bank 26/27 berbeda: \'+m)\n    f28=r.frame(up(28,\'ILS/01_prediksi_semua_jalur.csv\'),True);f28[\'run\']=0\n    a28=r.bank(up(28,\'ILS/06_bootstrap_paired_metrics.npz\'))\n    for i,raw in enumerate(a28[\'pathways\']):\n        pathway=normalize_route(str(raw));sub=f28.loc[f28.pathway.eq(raw)].copy();sub[\'pathway\']=pathway;b=metric_bank(a28,i)\n        if pathway in banks:\n            comparison_alias(r,old.loc[old.pathway.eq(pathway)],sub,\'28/\'+pathway,aliases)\n            if not np.array_equal(b,banks[pathway],equal_nan=True):raise RuntimeError(\'Bank alias 28 berbeda: \'+pathway)\n        else:\n            sub[\'_prediction_source\']=up(28,\'ILS/01_prediksi_semua_jalur.csv\');sub[\'_bank_source\']=up(28,\'ILS/06_bootstrap_paired_metrics.npz\');sub[\'seed\']=\'\'\n            old=pd.concat([old,sub],ignore_index=True);banks[pathway]=b\n    f29=r.frame(up(29,\'ILS/01_prediksi_per_peserta.csv\'),True);f29[\'run\']=0\n    a29=r.bank(up(29,\'ILS/06_bootstrap_paired_metrics.npz\'))\n    for i,k in enumerate(a29[\'k_values\']):\n        pathway=BASE if int(k)==16 else \'K%02d_PRIMARY\'%int(k)\n        sub=f29.loc[f29.k.astype(int).eq(int(k))].copy();sub[\'pathway\']=pathway;b=metric_bank(a29,i)\n        if pathway==BASE:\n            comparison_alias(r,old.loc[old.pathway.eq(pathway)],sub,\'29/k16\',aliases)\n            if not np.array_equal(b,banks[BASE],equal_nan=True):raise RuntimeError(\'Bank k16 berbeda dari CARE utama.\')\n        else:\n            sub[\'_prediction_source\']=up(29,\'ILS/01_prediksi_per_peserta.csv\');sub[\'_bank_source\']=up(29,\'ILS/06_bootstrap_paired_metrics.npz\');sub[\'seed\']=\'\'\n            old=pd.concat([old,sub],ignore_index=True);banks[pathway]=b\n    w=r.bank(up(27,\'ILS/09_bootstrap_weights.npz\'))\n    valid=np.asarray(w[\'valid_draws\'],bool);W=w[\'weights\'][valid]\n    if not valid.all():\n        if any(len(b)!=int(valid.sum()) for b in banks.values()):raise RuntimeError(\'Mask bootstrap lama tidak sejajar.\')\n    old[\'course\']=old.courseid;old[\'fold\']=0\n    for k in KEYS:old[\'predicted_\'+k]=directions(old,[\'predicted_\'+x for x in KEYS])[:,KEYS.index(k)]\n    contexts.append({\'cohort\':\'PENDIKOM\',\'protocol\':\'HOLDOUT_70_30\',\'frame\':old,\'banks\':banks,\'ids\':w[\'learner_ids\'].astype(str).tolist(),\n        \'weights\':W,\'inverse\':np.arange(len(w[\'learner_ids\'])),\'groups\':w[\'learner_ids\'].astype(str),\'reference\':refs,\'course_ref\':course_ref,\n        \'unit\':\'learner\',\'source_stage\':27,\'weight_source\':up(27,\'ILS/09_bootstrap_weights.npz\')})\n    for ds in [\'PENDIKOM\',\'LMS\']:\n        frames=[]\n        for route in [\'DIRECT\',\'NB\',\'DT\',\'SVM\']:\n            sub=r.frame(up(30,\'PREDIKSI/\'+ds+\'/\'+route+\'.csv\'),True)\n            sub[\'run\']=0;sub[\'pathway\']=normalize_route(route)\n            sub[\'_prediction_source\']=up(30,\'PREDIKSI/\'+ds+\'/\'+route+\'.csv\');sub[\'_bank_source\']=up(30,\'STATISTIK/LMS/02_bootstrap_metrics.npz\') if ds==\'LMS\' else up(28,\'ILS/06_bootstrap_paired_metrics.npz\');sub[\'seed\']=\'\'\n            for k in KEYS:sub[\'predicted_\'+k]=sub[k].astype(np.int8)\n            frames.append(sub)\n        if ds==\'PENDIKOM\':\n            for sub in frames:\n                target=old.loc[old.pathway.eq(sub.pathway.iloc[0])].set_index(\'identity\')\n                aligned=sub.set_index(\'identity\').loc[target.index]\n                if not np.array_equal(target[[\'predicted_\'+k for k in KEYS]].to_numpy(int),aligned[[\'predicted_\'+k for k in KEYS]].to_numpy(int)):\n                    raise RuntimeError(\'Alias 30/PENDIKOM berbeda: \'+sub.pathway.iloc[0])\n                aliases.append({\'alias\':\'30/PENDIKOM/\'+sub.pathway.iloc[0],\'rows\':len(sub),\'same_predictions_and_reference\':True,\'analysis_use\':\'one canonical copy\'})\n        else:\n            w=r.bank(up(30,\'STATISTIK/LMS/01_bootstrap_weights.npz\'));a=r.bank(up(30,\'STATISTIK/LMS/02_bootstrap_metrics.npz\'))\n            direct=frames[0];teacher=dict(zip(direct.identity,direct[KEYS].to_numpy(int).tolist()))\n            f=pd.concat(frames,ignore_index=True);f[\'course\']=f.identity.map(lms_course);f[\'fold\']=0\n            contexts.append({\'cohort\':\'LMS\',\'protocol\':\'HOLDOUT_70_30\',\'frame\':f,\'banks\':{normalize_route(str(p)):metric_bank(a,i) for i,p in enumerate(a[\'pathways\'])},\n                \'ids\':w[\'learner_ids\'].astype(str).tolist(),\'weights\':w[\'weights\'],\'inverse\':np.arange(len(w[\'learner_ids\'])),\'groups\':w[\'learner_ids\'].astype(str),\n                \'reference\':teacher,\'course_ref\':lms_course,\'unit\':\'learner\',\'source_stage\':30,\'weight_source\':up(30,\'STATISTIK/LMS/01_bootstrap_weights.npz\')})\n    for (ds,protocol),f in f32.groupby([\'cohort\',\'protocol\'],sort=False):\n        wrel=up(32,\'STATISTIK/\'+ds+\'/\'+protocol+\'/01_course_bootstrap_weights.npz\');w=r.bank(wrel);f=f.copy()\n        teacher=refs if ds==\'PENDIKOM\' else dict(zip(f.loc[f.pathway.eq(\'CARE-KM_DIRECT\')&f.run.eq(0),\'identity\'],f.loc[f.pathway.eq(\'CARE-KM_DIRECT\')&f.run.eq(0),KEYS].to_numpy(int).tolist()))\n        banks={}\n        for raw in f.pathway.unique():\n            rel=up(32,\'STATISTIK/\'+ds+\'/\'+protocol+\'/\'+digest(raw)[:16]+\'_bootstrap.npz\');a=r.bank(rel)\n            if list(a[\'metric_names\'])!=METRICS or list(a[\'dimension_names\'])!=KEYS:raise RuntimeError(\'Bank tahap 32 memiliki urutan berbeda.\')\n            b=np.concatenate([a[\'dimension_metrics\'],a[\'overall_metrics\'][:,None,:]],axis=1)\n            banks[normalize_route(raw)]=b\n        f[\'_prediction_source\']=up(32,\'PREDIKSI/01_semua_prediksi.csv.gz\')\n        f[\'_bank_source\']=f.pathway.map(lambda raw:up(32,\'STATISTIK/\'+ds+\'/\'+protocol+\'/\'+digest(raw)[:16]+\'_bootstrap.npz\'))\n        f[\'pathway\']=f.pathway.map(normalize_route)\n        for k in KEYS:f[\'predicted_\'+k]=f[k]\n        contexts.append({\'cohort\':ds,\'protocol\':protocol,\'frame\':f,\'banks\':banks,\'ids\':w[\'learner_ids\'].astype(str).tolist(),\'weights\':w[\'weights\'],\n            \'inverse\':w[\'learner_course_index\'].astype(int),\'groups\':w[\'course_ids\'].astype(str),\'reference\':teacher,\'course_ref\':course_ref if ds==\'PENDIKOM\' else lms_course,\n            \'unit\':\'course\',\'source_stage\':32,\'weight_source\':wrel})\n    return contexts,aliases\n\ndef evaluate_context(c,m):\n    ds,protocol=c[\'cohort\'],c[\'protocol\'];f=c[\'frame\'];ids=c[\'ids\'];n=len(ids)\n    if len(set(ids))!=n:raise RuntimeError(\'ID bootstrap ganda.\')\n    if set(f.identity)!=set(ids) or f.course.isna().any():raise RuntimeError(\'Prediksi/identitas/kursus tidak lengkap: \'+ds+\'/\'+protocol)\n    truth=np.asarray([c[\'reference\'][i] for i in ids],np.int8)\n    courses=np.asarray([c[\'course_ref\'][i] for i in ids],str)\n    if not (f.identity.map(c[\'course_ref\']).astype(str)==f.course.astype(str)).all():raise RuntimeError(\'Kursus prediksi berbeda dari metadata kanonik.\')\n    W=c[\'weights\'];inverse=c[\'inverse\'];groups=c[\'groups\'];B=len(W);g=len(groups)\n    if B!=10000 or W.shape!=(B,g) or not (W.sum(axis=1)==g).all() or (W<0).any():raise RuntimeError(\'Bobot bootstrap bukan protokol penuh atau tidak valid.\')\n    if c[\'unit\']==\'course\' and not np.array_equal(groups[inverse],courses):raise RuntimeError(\'Urutan kursus bootstrap tidak sejajar.\')\n    role=\'INDEPENDENT_ILS\' if ds==\'PENDIKOM\' else \'CARE_TEACHER_FIDELITY_NOT_ILS\'\n    common={\'cohort\':ds,\'protocol\':protocol,\'reference_role\':role,\'bootstrap_unit\':c[\'unit\']}\n    rows=[];conf=[];perrun=[];profile=[];refs=[];normalized=[];points={};banks=c[\'banks\']\n    for di,key in enumerate(KEYS):\n        t=truth[:,di];refs.append({**common,\'dimension\':key,\'n_test\':n,\'first\':int((t==1).sum()),\'second\':int((t==-1).sum()),\'reference_unresolved\':int((t==0).sum())})\n    for pathway,one in f.groupby(\'pathway\',sort=False):\n        runs=sorted(one.run.unique());arrays=[]\n        for run in runs:\n            sub=one.loc[one.run.eq(run)]\n            if len(sub)!=n or sub.identity.duplicated().any() or set(sub.identity)!=set(ids):raise RuntimeError(\'Prediksi per run tidak lengkap: \'+pathway)\n            aligned=sub.set_index(\'identity\').loc[ids]\n            pred=aligned[[\'predicted_\'+k for k in KEYS]].to_numpy(np.int8)\n            if not np.isin(pred,[-1,0,1]).all():raise RuntimeError(\'Prediksi di luar -1,0,1.\')\n            if all(\'reference_\'+k in aligned for k in KEYS):\n                observed=directions(aligned,[\'reference_\'+k for k in KEYS])\n                if not np.array_equal(observed,truth):raise RuntimeError(\'Referensi tidak sama antar jalur: \'+pathway)\n            arrays.append(pred)\n            nn=pd.DataFrame({\'cohort\':ds,\'protocol\':protocol,\'identity\':ids,\'course\':courses,\'fold\':aligned.fold.to_numpy(),\'pathway\':pathway,\'run\':int(run),\n                \'reference_role\':role,\'ILS_train_label_budget\':322 if pathway.startswith(\'ILS_\') else 0})\n            nn[\'seed\']=aligned.seed.to_numpy();nn[\'prediction_source\']=aligned[\'_prediction_source\'].to_numpy();nn[\'metric_bank_source\']=aligned[\'_bank_source\'].to_numpy();nn[\'weights_source\']=c[\'weight_source\']\n            for di,key in enumerate(KEYS):nn[\'reference_\'+key]=truth[:,di];nn[\'predicted_\'+key]=pred[:,di]\n            normalized.append(nn)\n        pred=np.stack(arrays);masks=m.confusion_masks(truth,pred);counts=masks.sum(axis=0);values=m.count_metrics(counts)\n        point=np.mean(values,axis=0);points[pathway]=np.mean(point,axis=0)\n        bank=banks[pathway]\n        if bank.shape!=(B,5,3):raise RuntimeError(\'Ukuran bank metrik berbeda: \'+pathway)\n        # Independently reconstruct selected existing draws from the six confusion cells.\n        gc=np.zeros((g,len(runs),4,6));np.add.at(gc,inverse,masks)\n        probes=np.unique([0,1,B//2,B-1]);bc=np.einsum(\'bg,grdc->brdc\',W[probes].astype(float),gc,optimize=True)\n        reproduced=np.mean(m.count_metrics(bc),axis=1);reproduced=np.concatenate([reproduced,np.mean(reproduced,axis=1)[:,None,:]],axis=1)\n        if not np.allclose(reproduced,bank[probes],rtol=0,atol=2e-13,equal_nan=True):raise RuntimeError(\'Bobot/prediksi/bank tidak sejajar: \'+ds+\'/\'+protocol+\'/\'+pathway)\n        for ri,run in enumerate(runs):\n            for di,key in enumerate(KEYS):\n                conf.append({**common,\'pathway\':pathway,\'run\':int(run),\'dimension\':key,**dict(zip(CELL_NAMES,counts[ri,di].astype(int)))})\n                perrun.append({**common,\'pathway\':pathway,\'run\':int(run),\'dimension\':key,**dict(zip(METRICS,values[ri,di]))})\n        for di,key in enumerate(KEYS+[\'OVERALL\']):\n            v=point[di] if di<4 else np.mean(point,axis=0)\n            row={**common,\'pathway\':pathway,\'dimension\':key,\'n_test\':n,\'n_courses\':len(set(courses)),\'n_runs\':len(runs),\n                \'prediction_coverage\':float((pred[:,:,di]!=0).mean()) if di<4 else float((pred!=0).mean()),\n                \'full_profile_coverage\':float((pred!=0).all(axis=2).mean()),\'reference_coverage\':float((truth[:,di]!=0).mean()) if di<4 else float((truth!=0).mean()),\n                \'ILS_train_label_budget\':322 if pathway.startswith(\'ILS_\') else 0}\n            for mi,name in enumerate(METRICS):\n                lo,hi,nv=interval(bank[:,di,mi]);sv=values[:,di,mi] if di<4 else np.mean(values[:,:,mi],axis=1)\n                row.update({name:float(v[mi]),name+\'_ci_low\':lo,name+\'_ci_high\':hi,name+\'_valid_bootstrap\':nv,name+\'_run_sd\':float(np.std(sv,ddof=1)) if len(sv)>1 else 0.})\n            rows.append(row)\n        # Exact four-direction agreement is an additional descriptor, not a replacement for mean BA.\n        eligible=(truth!=0).all(axis=1);correct=((pred==truth[None,:,:]).all(axis=2)&eligible[None,:]).mean(axis=0)\n        cg=np.bincount(inverse,weights=correct,minlength=g);eg=np.bincount(inverse,weights=eligible,minlength=g);ng=np.bincount(inverse,minlength=g)\n        numer=np.einsum(\'bg,g->b\',W,cg,optimize=False)\n        for name,denom,pointval in [(\'exact_profile_accuracy_known_reference\',np.einsum(\'bg,g->b\',W,eg,optimize=False),correct.sum()/eligible.sum() if eligible.sum() else np.nan),\n            (\'correct_four_directions_share_all_test\',np.einsum(\'bg,g->b\',W,ng,optimize=False),float(correct.mean()))]:\n            draws=np.divide(numer,denom,out=np.full(B,np.nan),where=denom>0);lo,hi,nv=interval(draws)\n            profile.append({**common,\'pathway\':pathway,\'metric\':name,\'value\':pointval,\'ci_low\':lo,\'ci_high\':hi,\'valid_bootstrap\':nv,\'n_test\':n,\'n_full_reference\':int(eligible.sum()),\'n_runs\':len(runs)})\n    pairs=[]\n    families={\'initializer\':[(BASE,m+\'|PRIMARY_MODE_FIRST\') for m in METHODS if m!=\'CARE-KM\'],\n        \'profile_operator\':[(BASE,p) for p in [\'DIRECT_BEHAVIORAL_RULES\',\'CARE-KM|MEMBER_ONLY\',\'CARE-KM|DIRECTION_CONSISTENT_MEMBER_FIRST\']],\n        \'classifier\':[(BASE,\'CDS_\'+p) for p in [\'NB\',\'DT\',\'SVM\']],\n        \'supervised_reference_different_label_budget\':[(\'ILS_\'+p,\'CDS_\'+p) for p in [\'NB\',\'DT\',\'SVM\']],\n        \'k_diagnostic\':[(p,BASE) for p in [\'K08_PRIMARY\',\'K12_PRIMARY\',\'K20_PRIMARY\',\'K24_PRIMARY\']]}\n    for family,contrasts in families.items():\n        present=[(a,b) for a,b in contrasts if a in banks and b in banks]\n        for a,b in present:\n            delta=banks[a][:,4,:]-banks[b][:,4,:]\n            for mi,name in enumerate(METRICS):\n                lo,hi,nv=interval(delta[:,mi]);flo,fhi,_=interval(delta[:,mi],.05/len(present))\n                pairs.append({**common,\'family\':family,\'first_pathway\':a,\'second_pathway\':b,\'metric\':\'mean_\'+name,\n                    \'difference_first_minus_second\':float(points[a][mi]-points[b][mi]),\'ci_low\':lo,\'ci_high\':hi,\'valid_bootstrap\':nv,\n                    \'family_contrasts\':len(present),\'family95_ci_low\':flo if name==\'balanced_accuracy\' else np.nan,\'family95_ci_high\':fhi if name==\'balanced_accuracy\' else np.nan,\n                    \'multiplicity\':\'Bonferroni percentile family interval, primary mean BA only; other CIs descriptive\'})\n    return {\'summary\':pd.DataFrame(rows),\'paired\':pd.DataFrame(pairs),\'confusions\':pd.DataFrame(conf),\'per_run\':pd.DataFrame(perrun),\n        \'exact_profiles\':pd.DataFrame(profile),\'reference_counts\':pd.DataFrame(refs),\'predictions\':pd.concat(normalized,ignore_index=True)}\n\ndef source_table(r,out,stage,rel,target,registry):\n    src=up(stage,rel);f=r.frame(src);save_csv(out/target,f)\n    registry.append({\'output\':target,\'upstream_script\':\'PYTHON/\'+STEMS[stage]+\'.ipynb\',\'source\':src,\'source_sha256\':r.hash(src),\'operation\':\'reuse completed results; no new fitting\'})\n    return f\n\ndef synthesis_tables(r,out,results,contexts,m,registry):\n    summary=results[\'summary\'];paired=results[\'paired\'];overall=summary.loc[summary.dimension.eq(\'OVERALL\')]\n    for n,rel,target in [(31,\'TABEL/01_lms_full_8_metode.csv\',\'TABEL/T04_LMS_full_8_metode.csv\'),\n        (31,\'TABEL/02_lms_resampling_8_metode.csv\',\'TABEL/T07_LMS_resampling_8_metode.csv\'),\n        (26,\'TABEL/02_initializer_stabilitas.csv\',\'TABEL/S01b_Stabilitas_initializer.csv\'),\n        (27,\'TABEL/03_stabilitas_aturan_utama.csv\',\'TABEL/S02b_Stabilitas_operator_profil.csv\'),\n        (29,\'TABEL/02_sensitivitas_k_stabilitas.csv\',\'TABEL/S04b_Stabilitas_k.csv\'),\n        (29,\'TABEL/03_diversitas_full_dan_train.csv\',\'TABEL/S04c_Diversitas_k.csv\'),\n        (27,\'01_ringkasan_pembentukan_profil.csv\',\'TABEL/S05_Konflik_semantik_semua_konteks.csv\'),\n        (31,\'TABEL/07_runtime_8_metode.csv\',\'TABEL/S06_Runtime_8_metode.csv\'),\n        (31,\'TABEL/08_failure_runtime.csv\',\'TABEL/S06b_Failure_runtime.csv\'),\n        (31,\'TABEL/09_runtime_per_seed.csv\',\'TABEL/S06c_Runtime_per_seed.csv\'),\n        (31,\'TABEL/06_failure_historis_8_metode.csv\',\'TABEL/S06d_Failure_kualitas.csv\'),\n        (30,\'TABEL/01_inference_per_cohort_batch.csv\',\'TABEL/S07_Waktu_inference.csv\'),\n        (30,\'TABEL/02_fidelity_terhadap_direct.csv\',\'TABEL/S07b_Ketidaksepakatan_classifier.csv\')]:source_table(r,out,n,rel,target,registry)\n    # Source25 is canonical; stage31 carries a byte-identical reporting copy.\n    bench=r.frame(up(25,\'02_ringkasan_8_metode.csv\'))\n    datasets=sorted(bench.loc[bench.scope.eq(\'BENCHMARK_FULL\'),\'dataset\'].unique())\n    if len(datasets)!=10:raise RuntimeError(\'Benchmark harus berisi sepuluh dataset.\')\n    notes=m.benchmark_statistics(bench,{\'resample_runs\':30,\'benchmark_stochastic_runs\':10},types.SimpleNamespace(DATASET_ORDER=datasets),out)\n    if notes:raise RuntimeError(\'Statistik benchmark belum lengkap: \'+str(notes))\n    for name,tgt in [(\'01_dataset_values_and_ranks.csv\',\'T08a_Nilai_dan_rank_per_dataset.csv\'),(\'02_average_ranks.csv\',\'T08_Rerata_rank.csv\'),\n        (\'03_friedman_kendall.csv\',\'T09_Friedman_Kendall.csv\'),(\'04_paired_wilcoxon_holm.csv\',\'T10_Wilcoxon_Holm.csv\'),(\'05_paired_differences_and_ranks.csv\',\'T10b_Selisih_berpasangan.csv\')]:\n        rel=\'STATISTIK_BENCHMARK/\'+name\n        a=r.frame(up(25,rel));b=r.frame(up(31,\'KUALITAS/\'+rel))\n        if not a.equals(b):raise RuntimeError(\'Salinan statistik benchmark 25/31 berbeda: \'+name)\n        current=pd.read_csv(out/rel,float_precision=\'round_trip\')\n        keys=[k for k in [\'source\',\'metric\',\'baseline\',\'dataset\',\'method\'] if k in a]\n        aa=a.sort_values(keys).reset_index(drop=True);bb=current.sort_values(keys).reset_index(drop=True)\n        pd.testing.assert_frame_equal(aa,bb,check_exact=False,rtol=1e-12,atol=1e-13,check_dtype=False)\n        save_csv(out/\'TABEL\'/tgt,current);registry.append({\'output\':\'TABEL/\'+tgt,\'upstream_script\':\'PYTHON/\'+STEMS[25]+\'.ipynb\',\'source\':up(25,rel),\'source_sha256\':r.hash(up(25,rel)),\'operation\':\'replay exact frozen statistics; source31 copy counted once\'})\n    # Supplementary tables keep protocols and reference roles explicit.\n    maps={\'S01_Initializer_vs_ILS.csv\':overall.loc[overall.cohort.eq(\'PENDIKOM\')&overall.pathway.isin([p+\'|PRIMARY_MODE_FIRST\' for p in METHODS])],\n        \'S02_Pembentukan_profil.csv\':overall.loc[overall.cohort.eq(\'PENDIKOM\')&overall.pathway.isin([BASE,\'DIRECT_BEHAVIORAL_RULES\',\'CARE-KM|MEMBER_ONLY\',\'CARE-KM|DIRECTION_CONSISTENT_MEMBER_FIRST\'])],\n        \'S03_Referensi_supervised_ILS.csv\':overall.loc[overall.cohort.eq(\'PENDIKOM\')&overall.protocol.eq(\'HOLDOUT_70_30\')&overall.pathway.isin([BASE,\'DIRECT_BEHAVIORAL_RULES\']+[\'CDS_\'+p for p in [\'NB\',\'DT\',\'SVM\']]+[\'ILS_\'+p for p in [\'NB\',\'DT\',\'SVM\',\'TRAIN_MAJORITY\']])],\n        \'S04_Sensitivitas_k.csv\':overall.loc[overall.cohort.eq(\'PENDIKOM\')&overall.protocol.eq(\'HOLDOUT_70_30\')&(overall.pathway.str.startswith(\'K\')|overall.pathway.eq(BASE))],\n        \'S08_Transportabilitas_Pendikom.csv\':overall.loc[overall.cohort.eq(\'PENDIKOM\')&~overall.protocol.eq(\'HOLDOUT_70_30\')],\n        \'S09_LMS_teacher_fidelity.csv\':overall.loc[overall.cohort.eq(\'LMS\')],\n        \'T12_Holdout_LMS_per_dimensi.csv\':summary.loc[summary.cohort.eq(\'LMS\')&summary.protocol.eq(\'HOLDOUT_70_30\')],\n        \'T13_Holdout_Pendikom_per_dimensi.csv\':summary.loc[summary.cohort.eq(\'PENDIKOM\')&summary.protocol.eq(\'HOLDOUT_70_30\')&summary.pathway.isin([BASE,\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\'])]}\n    for name,f in maps.items():save_csv(out/\'TABEL\'/name,f)\n    overlap=[]\n    for i,a in enumerate(contexts):\n        for b in contexts[i+1:]:\n            if a[\'cohort\']!=b[\'cohort\']:continue\n            ia,ib=set(a[\'ids\']),set(b[\'ids\']);overlap.append({\'cohort\':a[\'cohort\'],\'protocol_A\':a[\'protocol\'],\'protocol_B\':b[\'protocol\'],\'n_A\':len(ia),\'n_B\':len(ib),\'overlap\':len(ia&ib),\'unique_union\':len(ia|ib),\'interpretation\':\'overlapping cohort; do not pool protocols as independent datasets\'})\n    save_csv(out/\'TABEL/S10_Overlap_peserta_antar_protokol.csv\',pd.DataFrame(overlap))\n    # Preserve existing ablation/budget/mapping outputs without guessing by basename.\n    for src,name,script in [\n        (\'OUTPUT/EKSPERIMEN/LMS/17_Ablasi_Komponen_CARE_KM/19_component_ablation_summary.csv\',\'T05_Ablasi_komponen.csv\',\'17_Ablasi_Komponen_CARE_KM\'),\n        (\'OUTPUT/EKSPERIMEN/LMS/16_Sensitivitas_Candidate_Budget/18_q_sensitivity_summary.csv\',\'T06_Candidate_budget.csv\',\'16_Sensitivitas_Candidate_Budget\'),\n        (\'OUTPUT/EKSPERIMEN/SEMANTIK/11_Interpretasi_Semantik_FSLSM/08_cluster_to_fslsm_mapping.csv\',\'T11_Mapping_LMS_full.csv\',\'11_Interpretasi_Semantik_FSLSM\')]:\n        r.hash(\'PYTHON/\'+script+\'.ipynb\')\n        save_csv(out/\'TABEL\'/name,r.frame(src));registry.append({\'output\':\'TABEL/\'+name,\'upstream_script\':\'PYTHON/\'+script+\'.ipynb\',\'source\':src,\'source_sha256\':r.hash(src),\'operation\':\'reuse original completed result from numbered pipeline path\'})\n    save_csv(out/\'TABEL/S11_Selisih_berpasangan_semua_keluarga.csv\',paired)\n    # Retain evidence and method reconstruction resources as references, without executing them.\n    resources=[]\n    for p in sorted((r.root/\'OUTPUT/PRAPROSES\').rglob(\'*.csv\')):\n        if any(k in p.name.lower() for k in [\'specification\',\'threshold\',\'feature_freeze\',\'feature_audit\']):\n            rel=r.relative(p);resources.append({\'topic\':\'indicator reconstruction / train thresholds have separate states\',\'source\':rel,\'sha256\':r.hash(rel)})\n    for n in [26,27,28,29,30,31,32]:\n        for p in sorted((r.root/(\'OUTPUT/TAMBAHAN/\'+STEMS[n]+\'/PROTOKOL\')).glob(\'*.json\')):\n            rel=r.relative(p);resources.append({\'topic\':\'frozen protocol stage \'+str(n),\'source\':rel,\'sha256\':r.hash(rel)})\n    save_csv(out/\'AUDIT/04_Peta_protokol_dan_rekonstruksi.csv\',pd.DataFrame(resources))\n\ndef figures(out,summary,paired):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.titlesize\':12,\'axes.labelsize\':11,\'ytick.labelsize\':11,\'xtick.labelsize\':11,\'pdf.fonttype\':42})\n    for ds,protocol in summary[[\'cohort\',\'protocol\']].drop_duplicates().itertuples(index=False,name=None):\n        f=summary.loc[summary.cohort.eq(ds)&summary.protocol.eq(protocol)&summary.dimension.eq(\'OVERALL\')]\n        if ds==\'LMS\':f=f.loc[f.pathway.isin([BASE,\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\'])]\n        else:f=f.loc[f.pathway.isin([m+\'|PRIMARY_MODE_FIRST\' for m in METHODS]+[\'DIRECT_BEHAVIORAL_RULES\',\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\'])]\n        f=f.sort_values(\'balanced_accuracy\');v=f.balanced_accuracy.to_numpy()*100;lo=f.balanced_accuracy_ci_low.to_numpy()*100;hi=f.balanced_accuracy_ci_high.to_numpy()*100\n        labels=f.pathway.str.replace(\'|PRIMARY_MODE_FIRST\',\'\',regex=False).tolist()\n        fig,ax=plt.subplots(figsize=(10,max(4,len(f)*.38+1.6)));y=np.arange(len(f))\n        ax.hlines(y,lo,hi,color=\'#42637b\',linewidth=1.5);ax.plot(v,y,\'o\',color=\'#116c86\',ms=5);ax.set_yticks(y,labels)\n        ax.set_xlabel(\'Mean balanced accuracy (%)\' if ds==\'PENDIKOM\' else \'Mean balanced teacher fidelity (%)\')\n        ax.set_title(ds+\' / \'+protocol+\' — 95% paired-bootstrap marginal intervals\');ax.grid(axis=\'x\',alpha=.2);ax.set_xlim(max(0,float(np.nanmin(lo))-5),101)\n        fig.text(.5,.015,\'Unit: \'+(\'learner\' if protocol==\'HOLDOUT_70_30\' else \'course\')+\'; fixed fitted runs; protocols are not pooled.\',ha=\'center\',fontsize=11)\n        fig.tight_layout(rect=(0,.05,1,1));p=out/\'GAMBAR\'/(\'01_\'+ds+\'_\'+protocol);p.parent.mkdir(parents=True,exist_ok=True)\n        fig.savefig(str(p)+\'.png\',dpi=180);fig.savefig(str(p)+\'.pdf\');plt.close(fig)\n    f=paired.loc[paired.cohort.eq(\'PENDIKOM\')&paired.protocol.eq(\'COURSE_GROUP\')&paired.metric.eq(\'mean_balanced_accuracy\')]\n    fig,ax=plt.subplots(figsize=(11,7));y=np.arange(len(f));lo=f.family95_ci_low.to_numpy()*100;hi=f.family95_ci_high.to_numpy()*100\n    pretty={\'CARE-KM|MEMBER_ONLY\':\'Member-only\',\'CARE-KM|DIRECTION_CONSISTENT_MEMBER_FIRST\':\'Member-first\',\'DIRECT_BEHAVIORAL_RULES\':\'Behavioral rules\',\n        \'CDS_NB\':\'NB\',\'CDS_DT\':\'DT\',\'CDS_SVM\':\'SVM\'}\n    families={\'initializer\':\'Initializer\',\'profile_operator\':\'Profile rule\',\'classifier\':\'Classifier\'}\n    labels=[families.get(row.family,row.family)+\': \'+pretty.get(row.second_pathway,row.second_pathway.replace(\'|PRIMARY_MODE_FIRST\',\'\')) for row in f.itertuples()]\n    ax.hlines(y,lo,hi,color=\'#42637b\');ax.plot(f.difference_first_minus_second.to_numpy()*100,y,\'o\',color=\'#116c86\');ax.axvline(0,color=\'gray\',ls=\'--\');ax.set_yticks(y,labels)\n    ax.set_xlabel(\'CARE-KM minus comparator, mean BA (percentage points)\');ax.set_title(\'Pendikom course split — family-adjusted 95% intervals\')\n    fig.text(.5,.01,\'Bonferroni within each family; conditional course bootstrap, not new training repetitions.\',ha=\'center\',fontsize=11)\n    fig.tight_layout(rect=(0,.05,1,1));p=out/\'GAMBAR/02_Pendikom_course_paired\';fig.savefig(str(p)+\'.png\',dpi=180);fig.savefig(str(p)+\'.pdf\');plt.close(fig)\n\ndef report(out,results,statuses):\n    s=results[\'summary\'];p=results[\'paired\'];rows=[]\n    reviewer=[(\'R1-1\',\'Rekonstruksi indikator, thresholds, 18 fitur Pendikom\',\'AUDIT/04_Peta_protokol_dan_rekonstruksi.csv\',\'Perbarui uraian Metode dari definisi fitur dan state train; tabel hasil saja tidak menggantikan spesifikasi.\'),\n        (\'R1-2\',\'Determinisme, tie/empty rules, update dan kompleksitas\',\'AUDIT/02_status_sumber.json\',\'Jelaskan operasi dan kompleksitas dari kode asli; tidak menyimpulkan determinisme hanya dari skor yang sama.\'),\n        (\'R1-3\',\'Mode/member conflicts termasuk Strong\',\'TABEL/S05_Konflik_semantik_semua_konteks.csv\',\'Laporkan denominator, bobot anggota dan konflik Strong per konteks; magnitudo bukan skor ILS.\'),\n        (\'R1-4\',\'Tabel 12/13, counts, confusion dan paired CI\',\'EVALUASI/confusions.csv\',\'Tambahkan CI, counts dan ketidakpastian; mean sama tidak membuktikan ekuivalensi.\'),\n        (\'R1-5\',\'Benchmark, nilai/rank dataset, exact p, ties, Holm, effect size\',\'TABEL/T08a_Nilai_dan_rank_per_dataset.csv\',\'Gunakan 10 dataset sebagai unit; 30 resample tidak menjadi 300 dataset.\'),\n        (\'R1-6\',\'Runtime, hardware, pengulangan, failure\',\'TABEL/S06_Runtime_8_metode.csv\',\'Pisahkan init/end-to-end dari inference; cantumkan lingkungan sumber dan zero failures beserta attempts.\'),\n        (\'R1-7\',\'Initializer terbaru dengan protokol sama\',\'TABEL/S01_Initializer_vs_ILS.csv\',\'Identifikasi BK-Modes 2025 dan status referensi; delapan metode pada konteks yang tersedia.\'),\n        (\'R1-8\',\'Versi kode dan tabel ke script/output\',\'PROTOKOL/03_peta_tabel_ke_sumber.csv\',\'Publikasikan arsip/repository berversi setelah ditinjau; snapshot lokal bukan DOI/release publik.\'),\n        (\'R2-1\',\'Initializer ke mutu dan stabilitas semantik\',\'TABEL/S01b_Stabilitas_initializer.csv\',\'Pisahkan mutu internal clustering, validitas ILS dan stabilitas profil.\'),\n        (\'R2-2\',\'Direct assignment versus classifier\',\'TABEL/S07_Waktu_inference.csv\',\'Laporkan mismatch, biaya per batch dan per kohort; classifier tidak selalu lebih cepat.\'),\n        (\'R2-3\',\'Pembentukan profil dan supervised ILS\',\'TABEL/S03_Referensi_supervised_ILS.csv\',\'Bandingkan direct rules/operator; supervised memakai 322 label ILS dan bukan baseline tanpa label yang setara.\'),\n        (\'R2-4\',\'Sensitivitas jumlah cluster k\',\'TABEL/S04_Sensitivitas_k.csv\',\'Diagnostik grid k tetap; bukan tuning k dari test atau bukti optimum universal.\'),\n        (\'R2-5\',\'Seleksi kohort, split kursus/waktu\',\'TABEL/S08_Transportabilitas_Pendikom.csv\',\'Bahas 565→461, overlap, perbedaan train dan retrospektif; tidak mengklaim institusi baru.\'),\n        (\'R2-6\',\'Kontribusi operasi CARE-KM\',\'TABEL/T05_Ablasi_komponen.csv\',\'Kaitkan ablasi dan candidate budget dengan klaim operasi; jangan hanya mengandalkan ranking.\'),\n        (\'EDITOR\',\'Kode, README, revisi naskah dan surat jawaban\',\'PROTOKOL/03_peta_tabel_ke_sumber.csv\',\'Masih memerlukan penyuntingan naskah, surat jawaban, anonimisasi data dan publikasi repository oleh penulis.\')]\n    for code,topic,evidence,action in reviewer:rows.append({\'comment\':code,\'topic\':topic,\'evidence\':evidence,\'state\':\'evidence assembled; author interpretation/edit required\',\'remaining_action\':action})\n    save_csv(out/\'TABEL/S12_Matriks_bukti_reviewer.csv\',pd.DataFrame(rows))\n    claims=[]\n    for row in p.loc[p.metric.eq(\'mean_balanced_accuracy\')].to_dict(\'records\'):\n        a,b=row[\'family95_ci_low\'],row[\'family95_ci_high\']\n        conclusion=\'interval includes zero; no established directional difference\' if a<=0<=b else (\'first higher under this conditional interval\' if a>0 else \'second higher under this conditional interval\')\n        claims.append({**row,\'allowed_interpretation\':conclusion,\'limit\':\'Context-specific; conditional on stored fits. Absence of a difference is not equivalence.\'})\n    save_csv(out/\'TABEL/S13_Daftar_klaim_dan_batas.csv\',pd.DataFrame(claims))\n    lines=[\'# Analisis gabungan hasil eksperimen\',\'\',\n        \'Analisis ini membaca keluaran tahap 25–32 yang sudah selesai. Tidak menjalankan praproses, fitting, prediksi, eksperimen runtime, atau pemilihan seed/k baru.\',\n        \'\', \'## Protokol tetap dipisahkan\',\'\',\n        \'| Kohort | Protokol | N test unik | Referensi | Unit bootstrap |\',\'| --- | --- | ---: | --- | --- |\']\n    for ds,protocol in s[[\'cohort\',\'protocol\']].drop_duplicates().itertuples(index=False,name=None):\n        f=s.loc[s.cohort.eq(ds)&s.protocol.eq(protocol)].iloc[0];lines.append(f\'| {ds} | {protocol} | {int(f.n_test)} | {f.reference_role} | {f.bootstrap_unit} |\')\n    lines += [\'\', \'Peserta yang muncul pada beberapa protokol tidak dihitung sebagai kohort baru. ILS merupakan referensi independen Pendikom. Teacher LMS berasal dari CARE-KM yang dipasang pada train; direct teacher memiliki fidelity 100% pada arah yang resolved secara definisi.\',\n        \'\', \'## Aturan analisis\',\'\',\n        \'- Semua peserta test, termasuk unresolved, tetap disimpan. Prediksi 0 adalah abstention/kesalahan; referensi 0 dikeluarkan hanya dari metrik yang memerlukan kutub referensi dan coverage dilaporkan.\',\n        \'- Metrik dihitung per run lalu direratakan. Empat dimensi, peserta/kursus, metode dan run dipertahankan bersama pada bobot bootstrap sumber. Tidak ada majority vote atau best-seed selection.\',\n        \'- CI marginal 95% dan selisih berpasangan memakai bank sumber. Interval keluarga mean BA menggunakan Bonferroni per keluarga, bukan Holm. CI bersyarat pada fitted folds/runs; ketidakpastian fitting ulang tidak seluruhnya tercakup.\',\n        \'- Exact four-direction accuracy merupakan descriptor tambahan; tidak menggantikan rerata BA empat dimensi. Semua confusion counts per run adalah bilangan bulat.\',\n        \'- Benchmark: 10 dataset independen, mean 30 matched resamples per dataset; exact signed-rank sign enumeration dengan average tied ranks, zero tolerance 1e-12. Holm untuk tujuh kontras CARE-KM, terpisah bagi DBI dan SI.\',\n        \'- Runtime bersumber dari pengukuran tahap 31: delapan metode, 2.400 timing attempts, 80 warm-up awal, metode stokastik 10 seed × 3 blok. Waktu inference tahap 30 menggunakan definisi/batch berbeda. Tidak mengukur waktu baru di tahap 33.\',\n        \'\', \'## Ringkasan hasil utama Pendikom\',\'\',\n        \'| Protokol | Jalur | Mean BA (%) | CI 95% (%) | Label ILS train |\', \'| --- | --- | ---: | --- | ---: |\']\n    focus=[BASE,\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\',\'ILS_NB\',\'ILS_DT\',\'ILS_SVM\']\n    for row in s.loc[s.cohort.eq(\'PENDIKOM\')&s.dimension.eq(\'OVERALL\')&s.pathway.isin(focus)].to_dict(\'records\'):\n        label=row[\'pathway\'].replace(\'|\',\' / \')\n        lines.append(f"| {row[\'protocol\']} | {label} | {row[\'balanced_accuracy\']*100:.2f} | {row[\'balanced_accuracy_ci_low\']*100:.2f}–{row[\'balanced_accuracy_ci_high\']*100:.2f} | {row[\'ILS_train_label_budget\']} |")\n    lines += [\'\', \'## Penggunaan di naskah\',\'\',\n        \'T04/T07 mengganti perbandingan menjadi delapan metode. T08–T10 memuat statistik delapan metode. T12/T13 menambahkan hasil per dimensi dan CI, bukan menyamakan teacher LMS dengan ILS. T05/T06/T11 mempertahankan keluaran asli. S01–S13 menyediakan bukti tambahan; penempatan tabel utama/lampiran tetap keputusan penulis.\',\n        \'\', \'Jangan menulis CARE-KM terbaik untuk kelima metrik benchmark: periksa rank iterasi BK-Modes. Keunggulan ILS bergantung protokol; CI yang melintasi nol bukan bukti ekuivalensi. Supervised ILS memakai 322 label train; CDS/direct memakai nol label ILS, tetapi tetap mempelajari representasi/mapping dari data train.\',\n        \'\', \'565→461 adalah seleksi ketersediaan ILS, bukan bukti missingness acak. Split waktu Pendikom retrospektif; LMS temporal audit-only. Lima folds berbagi data train sehingga bukan lima dataset independen. Tidak ada institusi baru dalam hasil ini.\',\n        \'\', \'Lihat S12 untuk bukti per komentar dan pekerjaan narasi yang masih diperlukan; S13 untuk batas klaim per kontras. Tidak ada perubahan otomatis pada manuskrip atau publikasi repository.\',\n        \'\', \'## Versi dan keterlacakan\',\'\',\n        \'PROTOKOL/01_manifest_sumber.json mengunci isi sumber; 02_versi_analisis.json mengunci kode analisis dan versi pustaka; 03_peta_tabel_ke_sumber.csv menghubungkan tabel ke sumber dan script. Hash snapshot lokal bukan nomor release publik atau DOI.\',\n        \'\']\n    atomic(out/\'LAPORAN/01_Analisis_gabungan.md\',\'\\n\'.join(lines).encode())\n\ndef run(root,config,source,math_source):\n    if set(config)-{\'force\',\'figures\'}:raise ValueError(\'Konfigurasi tidak dikenal.\')\n    if not all(isinstance(config.get(k,v),bool) for k,v in [(\'force\',False),(\'figures\',True)]):raise ValueError(\'force/figures harus boolean.\')\n    r=Reader(root);out=r.path(\'OUTPUT/TAMBAHAN/\'+STAGE);out.mkdir(parents=True,exist_ok=True)\n    own=\'PYTHON/\'+STAGE+\'.ipynb\';r.hash(own)\n    print(\'Memeriksa kelengkapan dan isi keluaran tahap 25–32...\',flush=True)\n    statuses=verify_sources(r,out)\n    env={\'python\':platform.python_version(),\'packages\':{n:importlib.metadata.version(n) for n in [\'numpy\',\'pandas\',\'scipy\',\'matplotlib\']}}\n    code={\'engine\':hashlib.sha256(source.encode()).hexdigest(),\'math\':hashlib.sha256(math_source.encode()).hexdigest(),\'notebook\':r.registry[own]}\n    cp=out/\'KONTROL/cache_analisis.json\';cached={}\n    if cp.is_file():\n        try:\n            cached=json.loads(cp.read_text(encoding=\'utf-8\'))\n            if not isinstance(cached,dict):cached={}\n        except (ValueError,OSError):pass\n    # Re-hash every registered source; no filename-only cache is accepted.\n    cache_ok=not config.get(\'force\',False) and cached.get(\'status\')==\'selesai\' and isinstance(cached.get(\'sources\'),dict) and isinstance(cached.get(\'outputs\'),dict) and cached.get(\'code\')==code and cached.get(\'environment\')==env and cached.get(\'config\')=={\'figures\':config.get(\'figures\',True)}\n    if cache_ok:\n        for rel,want in cached.get(\'sources\',{}).items():\n            if r.hash(rel)!=want:cache_ok=False\n        expected=digest({k:v for k,v in cached.items() if k!=\'record_sha256\'})\n        cache_ok=cache_ok and r.registry==cached.get(\'sources\') and cached.get(\'record_sha256\')==expected and bool(cached.get(\'outputs\'))\n        if cache_ok:cache_ok=all(r.path(p).is_file() and filehash(r.path(p))==h for p,h in cached[\'outputs\'].items())\n    if cache_ok:\n        r.unchanged();print(\'Cache analisis 33 valid: isi sumber, kode, lingkungan dan seluruh hasil sama.\',flush=True)\n        return {\'status\':\'CACHE_VALID\',\'output\':str(out),\'summary\':pd.read_csv(out/\'EVALUASI/summary.csv\'),\'paired\':pd.read_csv(out/\'EVALUASI/paired.csv\')}\n    print(\'Menggabungkan prediksi dan bank ketidakpastian tanpa fitting ulang...\',flush=True)\n    m=types.ModuleType(\'_math33\');m.__dict__.update({\'np\':np,\'pd\':pd,\'itertools\':itertools,\'save_csv\':save_csv,\'save_json\':save_json})\n    exec(compile(math_source,\'_math33\',\'exec\'),m.__dict__)\n    contexts,aliases=canonical_contexts(r);parts=[]\n    for c in contexts:\n        print(\'Analisis:\',c[\'cohort\'],c[\'protocol\'],\'N =\',len(c[\'ids\']),flush=True);parts.append(evaluate_context(c,m))\n    results={key:pd.concat([p[key] for p in parts],ignore_index=True) for key in parts[0]}\n    for key,f in results.items():save_csv(out/(\'PREDIKSI/01_prediksi_kanonik.csv.gz\' if key==\'predictions\' else \'EVALUASI/\'+key+\'.csv\'),f)\n    save_csv(out/\'AUDIT/03_Alias_dihitung_satu_kali.csv\',pd.DataFrame(aliases))\n    registry=[];synthesis_tables(r,out,results,contexts,m,registry)\n    if config.get(\'figures\',True):figures(out,results[\'summary\'],results[\'paired\'])\n    else:\n        for p in (out/\'GAMBAR\').glob(\'*\'):\n            if p.suffix.lower() in (\'.png\',\'.pdf\') and p.is_file() and p.resolve().is_relative_to(out.resolve()):p.unlink()\n    report(out,results,statuses)\n    manifest=dict(sorted(r.registry.items()));snapshot=digest(manifest)\n    save_json(out/\'PROTOKOL/01_manifest_sumber.json\',manifest)\n    save_json(out/\'PROTOKOL/02_versi_analisis.json\',{\'stage\':STAGE,\'code\':code,\'environment\':env,\'source_snapshot_sha256\':snapshot,\n        \'public_release_url\':None,\'public_release_id\':None,\'git_commit\':None,\'version_scope\':\'local content-addressed analysis snapshot; publication remains an author action\'})\n    srcrows=[]\n    for c in contexts:\n        for (path,pred,bank),one in c[\'frame\'].groupby([\'pathway\',\'_prediction_source\',\'_bank_source\'],sort=False):\n            stage=next(n for n,stem in STEMS.items() if \'/\'+stem+\'/\' in pred)\n            srcrows.append({\'cohort\':c[\'cohort\'],\'protocol\':c[\'protocol\'],\'pathway\':path,\'upstream_script\':\'PYTHON/\'+STEMS[stage]+\'.ipynb\',\n                \'prediction_source\':pred,\'prediction_sha256\':r.hash(pred),\'metric_bank_source\':bank,\'metric_bank_sha256\':r.hash(bank),\n                \'weight_source\':c[\'weight_source\'],\'weight_sha256\':r.hash(c[\'weight_source\']),\'n_test\':len(c[\'ids\']),\'bootstrap_unit\':c[\'unit\']})\n    save_csv(out/\'PROTOKOL/04_Sumber_evaluasi_per_protokol.csv\',pd.DataFrame(srcrows))\n    # Each table is linked both to the analysis script and the immutable upstream snapshot.\n    by_output={x[\'output\']:x for x in registry}\n    for p in sorted((out/\'TABEL\').glob(\'*.csv\')):\n        target=p.relative_to(out).as_posix();row=by_output.get(target,{\'output\':target,\'upstream_script\':\'; \'.join(\'PYTHON/\'+stem+\'.ipynb\' for stem in STEMS.values()),\n            \'source\':\'PROTOKOL/04_Sumber_evaluasi_per_protokol.csv\',\'source_sha256\':filehash(out/\'PROTOKOL/04_Sumber_evaluasi_per_protokol.csv\'),\'operation\':\'canonical predictions / paired banks / report synthesis; full dependency snapshot in manifest\'})\n        row.update({\'analysis_script\':own,\'analysis_code_sha256\':code[\'notebook\'].split(\':\')[-1],\'source_snapshot_sha256\':snapshot,\'output_sha256\':filehash(p)});registry.append(row) if target not in by_output else None\n    save_csv(out/\'PROTOKOL/03_peta_tabel_ke_sumber.csv\',pd.DataFrame(registry))\n    r.unchanged()\n    status={\'status\':\'selesai\',\'stage\':STAGE,\'source_stages\':list(STEMS),\'n_contexts\':len(contexts),\'n_source_files_verified\':len(r.registry),\n        \'n_canonical_prediction_rows\':len(results[\'predictions\']),\'new_model_fits\':0,\'new_predictions\':0,\'new_timing_measurements\':0,\'old_outputs_modified\':False,\'source_snapshot_sha256\':snapshot,\'finished_utc\':datetime.now(timezone.utc).isoformat()}\n    save_json(out/\'KONTROL/status_eksekusi.json\',status)\n    outputs={r.relative(p):filehash(p) for p in out.rglob(\'*\') if p.is_file() and p!=cp and not p.name.endswith(\'.tmp\')}\n    cache={\'status\':\'selesai\',\'code\':code,\'environment\':env,\'config\':{\'figures\':config.get(\'figures\',True)},\'sources\':dict(sorted(r.registry.items())),\'outputs\':outputs}\n    cache[\'record_sha256\']=digest(cache);save_json(cp,cache)\n    print(\'Selesai. Tidak ada fitting atau pengukuran waktu baru.\',flush=True)\n    return {\'status\':status,\'output\':str(out),\'summary\':results[\'summary\'],\'paired\':results[\'paired\']}\n'

_analysis33 = types.ModuleType('_analysis33')
exec(compile(RUNTIME33_SOURCE, '_analysis33', 'exec'), _analysis33.__dict__)


In [5]:
HASIL_33 = _analysis33.run(ROOT_33, KONFIGURASI_33, RUNTIME33_SOURCE, MATH33_SOURCE)
display(HASIL_33['summary'].loc[HASIL_33['summary'].dimension.eq('OVERALL')])
display(HASIL_33['paired'].loc[HASIL_33['paired'].metric.eq('mean_balanced_accuracy')])
print('Status:', HASIL_33['status'])
print('Folder hasil:', HASIL_33['output'])


Memeriksa kelengkapan dan isi keluaran tahap 25–32...
Memeriksa integritas sumber: 1000
Memeriksa integritas sumber: 2000
Memeriksa integritas sumber: 3000
Memeriksa integritas sumber: 4000
Memeriksa integritas sumber: 5000
Memeriksa integritas sumber: 6000
Memeriksa integritas sumber: 7000
Memeriksa integritas sumber: 8000
Memeriksa integritas sumber: 9000
Memeriksa integritas sumber: 10000
Memeriksa integritas sumber: 11000
Menggabungkan prediksi dan bank ketidakpastian tanpa fitting ulang...
Analisis: PENDIKOM HOLDOUT_70_30 N = 139
Analisis: LMS HOLDOUT_70_30 N = 1998
Analisis: PENDIKOM COURSE_GROUP N = 461
Analisis: PENDIKOM TEMPORAL N = 115
Analisis: LMS COURSE_GROUP N = 6658
Selesai. Tidak ada fitting atau pengukuran waktu baru.


,cohort,protocol,reference_role,bootstrap_unit,pathway,dimension,n_test,n_courses,n_runs,prediction_coverage,...,balanced_accuracy,balanced_accuracy_ci_low,balanced_accuracy_ci_high,balanced_accuracy_valid_bootstrap,balanced_accuracy_run_sd,macro_f1,macro_f1_ci_low,macro_f1_ci_high,macro_f1_valid_bootstrap,macro_f1_run_sd
4,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,Random|PRIMARY_MODE_FIRST,OVERALL,139,34,15,0.993885,...,0.721997,0.686027,0.757130,10000,0.039195,0.718101,0.673600,0.754848,10000,0.042543
9,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,Random|MEMBER_ONLY,OVERALL,139,34,15,0.990168,...,0.722017,0.685820,0.757311,10000,0.040984,0.719683,0.674673,0.756566,10000,0.043074
14,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,Random|DIRECTION_CONSISTENT_MEMBER_FIRST,OVERALL,139,34,15,0.993885,...,0.723774,0.687713,0.759168,10000,0.040533,0.720087,0.675060,0.756942,10000,0.042871
19,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,Huang|PRIMARY_MODE_FIRST,OVERALL,139,34,15,0.996283,...,0.701004,0.668817,0.732012,10000,0.029278,0.693733,0.656375,0.724123,10000,0.037411
24,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,Huang|MEMBER_ONLY,OVERALL,139,34,15,0.995084,...,0.700357,0.667929,0.731819,10000,0.033531,0.695598,0.657266,0.726810,10000,0.042319
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,LMS,COURSE_GROUP,CARE_TEACHER_FIDELITY_NOT_ILS,course,HR-2023|PRIMARY_MODE_FIRST,OVERALL,6658,171,15,0.999459,...,0.695383,0.678426,0.709140,10000,0.014078,0.692497,0.674029,0.706475,10000,0.011537
394,LMS,COURSE_GROUP,CARE_TEACHER_FIDELITY_NOT_ILS,course,DIRECT_BEHAVIORAL_RULES,OVERALL,6658,171,1,0.842896,...,0.645647,0.627880,0.660120,10000,0.000000,0.690577,0.672915,0.703802,10000,0.000000
399,LMS,COURSE_GROUP,CARE_TEACHER_FIDELITY_NOT_ILS,course,CDS_NB,OVERALL,6658,171,1,1.000000,...,0.819324,0.798882,0.836488,10000,0.000000,0.808990,0.788037,0.827067,10000,0.000000
404,LMS,COURSE_GROUP,CARE_TEACHER_FIDELITY_NOT_ILS,course,CDS_DT,OVERALL,6658,171,1,1.000000,...,0.855590,0.836120,0.871554,10000,0.000000,0.853128,0.834510,0.868639,10000,0.000000


,cohort,protocol,reference_role,bootstrap_unit,family,first_pathway,second_pathway,metric,difference_first_minus_second,ci_low,ci_high,valid_bootstrap,family_contrasts,family95_ci_low,family95_ci_high,multiplicity
1,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,initializer,CARE-KM|PRIMARY_MODE_FIRST,Random|PRIMARY_MODE_FIRST,mean_balanced_accuracy,0.090929,0.052506,0.129437,10000,7,0.037476,0.146988,"Bonferroni percentile family interval, primary..."
4,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,initializer,CARE-KM|PRIMARY_MODE_FIRST,Huang|PRIMARY_MODE_FIRST,mean_balanced_accuracy,0.111922,0.071662,0.152275,10000,7,0.055229,0.168376,"Bonferroni percentile family interval, primary..."
7,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,initializer,CARE-KM|PRIMARY_MODE_FIRST,Cao|PRIMARY_MODE_FIRST,mean_balanced_accuracy,0.043436,0.011601,0.079321,10000,7,0.000776,0.096821,"Bonferroni percentile family interval, primary..."
10,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,initializer,CARE-KM|PRIMARY_MODE_FIRST,Bai|PRIMARY_MODE_FIRST,mean_balanced_accuracy,0.068729,0.027998,0.115910,10000,7,0.013593,0.137375,"Bonferroni percentile family interval, primary..."
13,PENDIKOM,HOLDOUT_70_30,INDEPENDENT_ILS,learner,initializer,CARE-KM|PRIMARY_MODE_FIRST,K-Modes++|PRIMARY_MODE_FIRST,mean_balanced_accuracy,0.071445,0.035489,0.109034,10000,7,0.021824,0.125626,"Bonferroni percentile family interval, primary..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,LMS,COURSE_GROUP,CARE_TEACHER_FIDELITY_NOT_ILS,course,profile_operator,CARE-KM|PRIMARY_MODE_FIRST,CARE-KM|MEMBER_ONLY,mean_balanced_accuracy,0.007313,0.003771,0.012595,10000,3,0.003166,0.014153,"Bonferroni percentile family interval, primary..."
175,LMS,COURSE_GROUP,CARE_TEACHER_FIDELITY_NOT_ILS,course,profile_operator,CARE-KM|PRIMARY_MODE_FIRST,CARE-KM|DIRECTION_CONSISTENT_MEMBER_FIRST,mean_balanced_accuracy,0.007313,0.003771,0.012595,10000,3,0.003166,0.014153,"Bonferroni percentile family interval, primary..."
178,LMS,COURSE_GROUP,CARE_TEACHER_FIDELITY_NOT_ILS,course,classifier,CARE-KM|PRIMARY_MODE_FIRST,CDS_NB,mean_balanced_accuracy,0.180676,0.163512,0.201118,10000,3,0.159917,0.205952,"Bonferroni percentile family interval, primary..."
181,LMS,COURSE_GROUP,CARE_TEACHER_FIDELITY_NOT_ILS,course,classifier,CARE-KM|PRIMARY_MODE_FIRST,CDS_DT,mean_balanced_accuracy,0.144410,0.128446,0.163880,10000,3,0.124779,0.167882,"Bonferroni percentile family interval, primary..."


Status: {'status': 'selesai', 'stage': '33_Analisis_Gabungan_Hasil', 'source_stages': [25, 26, 27, 28, 29, 30, 31, 32], 'n_contexts': 5, 'n_source_files_verified': 11852, 'n_canonical_prediction_rows': 542728, 'new_model_fits': 0, 'new_predictions': 0, 'new_timing_measurements': 0, 'old_outputs_modified': False, 'source_snapshot_sha256': 'a6f1077d9ac36b2efaf391772973ced4f11b54d2e566c63a0d367469cab3b95f', 'finished_utc': '2026-11-07T14:42:53.541029+00:00'}
Folder hasil: /content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/33_Analisis_Gabungan_Hasil


## Keluaran dan tindak lanjut

- `TABEL/`: T04–T13 dan tabel tambahan S01–S13, dipisahkan menurut konteks dan
  referensi. Ini usulan penempatan tabel; nomor final naskah ditentukan penulis.
- `EVALUASI/`: hasil per dimensi/run, confusion counts, reference counts,
  exact four-direction accuracy, dan selisih berpasangan beserta CI.
- `PREDIKSI/`: satu bank prediksi kanonik yang sudah tersedia; tidak memanggil
  model untuk membuat prediksi baru.
- `GAMBAR/`: PNG/PDF hasil utama per protokol dan kontras kursus Pendikom.
- `AUDIT/`: integritas sumber, status, alias yang dihitung sekali, dan peta
  protokol/rekonstruksi indikator.
- `LAPORAN/01_Analisis_gabungan.md`: ringkasan hasil, penggunaan di naskah,
  batas interpretasi, dan pekerjaan narasi yang masih diperlukan.
- `PROTOKOL/`: manifest sumber, versi lokal, peta tabel ke script/sumber/hash.
- `KONTROL/`: cache isi dan status analisis 33.

Waktu tahap 31 merupakan pengukuran delapan metode dengan lingkungan dan
definisi bersama. Waktu inference tahap 30 memiliki definisi/batch berbeda dan
tidak ditempelkan pada model baru tahap 32. Jangan menyatakan classifier selalu
lebih cepat, CARE-KM selalu unggul, CI melintasi nol berarti ekuivalen, atau
validasi kursus/waktu merupakan institusi baru. Supervised ILS memakai 322
label train sehingga harus dibedakan dari metode dengan nol label ILS.
